<a href="https://colab.research.google.com/github/simecek/dspracticum2026/blob/main/lesson04/gpt2_inside.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 04: Inside GPT-2

Last week, in the 3Blue1Brown video, you saw in pictures how a transformer works. Today we open up a real one: **GPT-2**, released by OpenAI in 2019. ChatGPT, Claude, Llama and Gemma are its direct descendants, and inside they have changed surprisingly little.

**What you will do:**
- turn text into **tokens**, and rebuild GPT-2's tokenizer yourself
- look at GPT-2's **embeddings**: the vectors that represent tokens
- rebuild a **transformer block** in a few lines of PyTorch and check that it gives *the same numbers* as the real GPT-2
- look at what the **attention heads** look at
- turn vectors back into **next-token probabilities** and generate text
- **train** your own mini GPT on Shakespeare, using the same building blocks

Throughout the notebook we follow one example: the prompt **"The capital of France is"**, and what GPT-2 thinks comes next.

**Contents:** 0. Setup · 1. Text is just bytes · 2. Byte Pair Encoding by hand · 3. GPT-2's tokenizer · 4. Meet GPT-2 · 5. Embeddings · 6. A transformer block by hand · 7. What do attention heads look at? · 8. From vectors back to text · 9. Bonus: the logit lens · 10. Training a mini GPT · 11. Summary

**Before you start:** *Runtime → Change runtime type → T4 GPU*. Sections 1-9 run fine on a CPU, but the training in section 10 needs a GPU, and changing the runtime later erases everything you have already run.

---
## 0. Setup

In [ ]:
import math
import urllib.request

import matplotlib.pyplot as plt
import regex as re      # like Python's built-in "re", but understands Unicode classes such as "any letter"
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import tiktoken     # OpenAI's tokenizer library
except ImportError:
    !pip install -q tiktoken
    import tiktoken

In [ ]:
# @title Helper functions (just run this cell)
import html
from IPython.display import HTML, display

COLORS = ["#fde68a", "#bfdbfe", "#bbf7d0", "#fbcfe8", "#ddd6fe", "#fed7aa"]

def show_tokens(pieces):
    """Show text pieces (str or bytes) in alternating colors, like the Tiktokenizer app."""
    spans = []
    for i, piece in enumerate(pieces):
        if isinstance(piece, bytes):
            piece = piece.decode("utf-8", errors="replace")
        piece = html.escape(piece).replace("\n", "↵\n")
        spans.append(f'<span style="background:{COLORS[i % len(COLORS)]}; color:black; border-radius:3px">{piece}</span>')
    display(HTML(f'<pre style="font-size:15px; line-height:1.8; white-space:pre-wrap; margin:0">{"".join(spans)}</pre>'
                 f'<div style="font-size:12px; margin-bottom:8px">{len(pieces)} tokens</div>'))

def plot_attention(weights, tokens, title=""):
    """Heatmap of one attention head: each row is a token, the columns show where it looks."""
    weights = weights.detach().cpu()
    size = min(9, max(4, 0.35 * len(tokens)))
    plt.figure(figsize=(size + 1, size))
    plt.imshow(weights, cmap="Blues", vmin=0, vmax=1)
    plt.xticks(range(len(tokens)), tokens, rotation=90)
    plt.yticks(range(len(tokens)), tokens)
    plt.xlabel("... looks at this token")
    plt.ylabel("this token ...")
    plt.title(title)
    plt.colorbar(fraction=0.046)
    plt.show()

def plot_heads(weights, title=""):
    """All heads of one block as small heatmaps. weights: (heads, tokens, tokens)."""
    n = weights.shape[0]
    fig, axes = plt.subplots(2, n // 2, figsize=(2 * n // 2, 5))
    for h, ax in enumerate(axes.flat):
        ax.imshow(weights[h].detach().cpu(), cmap="Blues", vmin=0, vmax=1)
        ax.set_title(f"head {h}", fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

def plot_head_scores(score, title):
    """A (blocks x heads) table of numbers as a heatmap."""
    plt.figure(figsize=(6, 4.5))
    plt.imshow(score.cpu(), cmap="viridis", vmin=0, vmax=1)
    plt.xlabel("head")
    plt.ylabel("block")
    plt.xticks(range(score.shape[1]))
    plt.yticks(range(score.shape[0]))
    plt.title(title)
    plt.colorbar()
    plt.show()

---
## 1. Text is just bytes

A neural network only understands numbers, so first we need to turn text into numbers. Computers already do that: every character has a number in the **Unicode** table.

In [ ]:
text = "Ahoj! Příliš žluťoučký kůň 🐴"
print([ord(ch) for ch in text])      # ord() gives the Unicode number of a character

Unicode has about 150,000 characters, far too many for a vocabulary. To store them, computers use **UTF-8**, which writes every character as **1 to 4 bytes** (numbers 0-255). English letters take 1 byte, Czech letters 2, the euro sign 3, emoji 4:

In [ ]:
for ch in "aAřž€🐴":
    print(ch, "  Unicode:", ord(ch), "  UTF-8 bytes:", list(ch.encode("utf-8")))

In [ ]:
data = text.encode("utf-8")
print(list(data))
print(len(text), "characters →", len(data), "bytes")
print(data.decode("utf-8"))           # and back to text

So we could feed a network **bytes**: a vocabulary of only 256 "tokens", and *any* text in any language can be written with them. The problem is that the sequences get long (a Czech letter takes 2 bytes, an emoji 4), and the network would waste its effort on spelling.

**Tokenization** is the compromise: glue frequent sequences of bytes into bigger **tokens**.

**Try it:** How many bytes does your name take? And a sentence in Chinese or Ukrainian?

In [ ]:
# your code here

---
## 2. Byte Pair Encoding (BPE) by hand

GPT-2 builds its tokens with **Byte Pair Encoding**, a very simple algorithm:
1. Start with the text as a list of bytes.
2. Find the **pair of neighbours** that occurs most often.
3. Replace it everywhere with a **new token** (256, 257, ...).
4. Repeat.

We need two small functions. `get_stats` counts the pairs:

In [ ]:
def get_stats(ids):
    """Count how often each pair of neighbours occurs."""
    counts = {}
    for pair in zip(ids, ids[1:]):          # (ids[0], ids[1]), (ids[1], ids[2]), ...
        counts[pair] = counts.get(pair, 0) + 1
    return counts

ids = list("aaabdaaabac".encode("utf-8"))
print(ids)
get_stats(ids)

The most frequent pair is `(97, 97)`, i.e. `"aa"`. `merge` replaces it with a new token, 256:

In [ ]:
def merge(ids, pair, new_id):
    """Replace every occurrence of `pair` in `ids` with `new_id`."""
    result = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i + 1]) == pair:
            result.append(new_id)
            i += 2                          # skip both members of the pair
        else:
            result.append(ids[i])
            i += 1
    return result

merge(ids, (97, 97), 256)

11 tokens became 9. Now let's try real text. We download **tiny Shakespeare** (1 MB of Shakespeare's plays). We will also use it to train our own GPT in section 10.

In [ ]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "input.txt")
with open("input.txt", encoding="utf-8") as f:
    shakespeare = f.read()

print(f"{len(shakespeare):,} characters")
print(shakespeare[:300])

**Training** a BPE tokenizer means repeating `get_stats` + `merge` and remembering every merge. Our pure-Python version is slow, so we use only the first 20,000 characters and do 30 merges:

In [ ]:
def train_bpe(text, num_merges):
    ids = list(text.encode("utf-8"))
    merges = {}                                    # (id1, id2) -> new id
    vocab = {i: bytes([i]) for i in range(256)}    # id -> the bytes it stands for
    for k in range(num_merges):
        stats = get_stats(ids)
        pair = max(stats, key=stats.get)           # the most frequent pair
        new_id = 256 + k
        ids = merge(ids, pair, new_id)
        merges[pair] = new_id
        vocab[new_id] = vocab[pair[0]] + vocab[pair[1]]
        a, b, new = (vocab[i].decode("utf-8", errors="replace") for i in (*pair, new_id))
        print(f"merge {k + 1:2}:  {a!r:8} + {b!r:8} → {new!r:10} ({stats[pair]} times)")
    return merges, vocab

sample = shakespeare[:20_000]
merges, vocab = train_bpe(sample, 30)

First the algorithm merges common letter pairs, many of them with a space (`'e '`, `'t '`, `'s '`): **the space becomes part of the token**. Merge 19 is already a whole word: `'th'` + `'e '` → `'the '`.

To **encode** new text, we apply the merges in the same order as in training. (Later merges are built from earlier ones, so the order matters.)

In [ ]:
def bpe_encode(text, merges):
    ids = list(text.encode("utf-8"))
    while len(ids) >= 2:
        stats = get_stats(ids)
        # among the pairs in the text, pick the one that was learned EARLIEST
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break                                   # nothing left to merge
        ids = merge(ids, pair, merges[pair])
    return ids

def bpe_decode(ids, vocab):
    return b"".join(vocab[i] for i in ids).decode("utf-8", errors="replace")

sentence = "To be, or not to be, that is the question."
ids = bpe_encode(sentence, merges)
print(len(sentence.encode("utf-8")), "bytes →", len(ids), "tokens")
show_tokens([vocab[i] for i in ids])
print(bpe_decode(ids, vocab))

**Try it:** Train with 100 merges instead of 30 (it takes a few seconds). Which whole words become tokens? How well does it work on a Czech sentence?

GPT-2 did exactly the same, only on 40 GB of text and with **50,000 merges**. Its vocabulary is 256 bytes + 50,000 merges + 1 special token = **50,257 tokens**.

In [ ]:
# your code here

---
## 3. GPT-2's tokenizer

OpenAI's library `tiktoken` contains the real GPT-2 tokenizer:

In [ ]:
enc = tiktoken.get_encoding("gpt2")
print("vocabulary size:", enc.n_vocab)

ids = enc.encode("The capital of France is")
print(ids)
print([enc.decode([i]) for i in ids])

### 3a. Rebuilding it ourselves

Can we reproduce it with our own code? We need two ingredients.

**Ingredient 1: the 50,000 merges.** `tiktoken` stores them as a dictionary *bytes → token id*. The id is also the **order** in which the merges were learned: id 256 was the first merge, id 50,255 the last.

In [ ]:
ranks = enc._mergeable_ranks        # starts with "_" = meant to be private, but it is exactly what we need
print(len(ranks), "entries")

print("first merges: ", [enc.decode([i]) for i in range(256, 271)])
print("last merges:  ", [enc.decode([i]) for i in range(50245, 50256)])
print("longest tokens:", [t.decode("utf-8", errors="replace") for t in sorted(ranks, key=len)[-6:]])

The very first merge GPT-2 learned was `' t'`, and `' the'` came 7th. Notice that GPT-2 puts the space at the **start** of a word (`' the'`), while our Shakespeare tokenizer put it at the end (`'the '`). The reason is the pre-split, see below. The longest tokens are lines of dots, `=` and `_`, plus a piece of garbled text (`ÃÂÃÂ...`, a broken character encoding) that was common enough on the web to get its own token.

**Ingredient 2: a pre-split.** Before merging, GPT-2 splits the text into chunks with a regular expression: words (with the space in front of them), numbers, punctuation and whitespace. Merges only happen *inside* a chunk, so `"dog."`, `"dog!"` and `"dog?"` never become three separate tokens.

(You don't need to read the regex. `\p{L}` means "any letter, in any alphabet" and `\p{N}` "any digit". Just look at the output.)

In [ ]:
gpt2_pattern = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

print(gpt2_pattern.findall("Hello've world123 how's are you!!!?"))

**The algorithm:** in each chunk, start from single bytes and keep merging the neighbouring pair that was learned **earliest** (lowest id), until no pair can be merged. Watch the word `" tokenization"`:

In [ ]:
def encode_chunk(chunk, verbose=False):
    parts = [bytes([b]) for b in chunk]                 # start: one part per byte
    while len(parts) > 1:
        # all neighbouring pairs that GPT-2 can merge, with their rank (= order of learning)
        candidates = [(ranks[parts[i] + parts[i + 1]], i)
                      for i in range(len(parts) - 1)
                      if parts[i] + parts[i + 1] in ranks]
        if not candidates:
            break
        rank, i = min(candidates)                       # the earliest-learned merge wins
        parts = parts[:i] + [parts[i] + parts[i + 1]] + parts[i + 2:]
        if verbose:
            print([p.decode("utf-8", errors="replace") for p in parts])
    return [ranks[p] for p in parts]

def our_encode(text):
    ids = []
    for chunk in gpt2_pattern.findall(text):
        ids += encode_chunk(chunk.encode("utf-8"))
    return ids

encode_chunk(" tokenization".encode("utf-8"), verbose=True)

Now the test: does our tokenizer give the same tokens as the real one?

In [ ]:
texts = [
    "The capital of France is",
    "Hlavní město Francie je Paříž.",
    "안녕하세요 👋 (hello in Korean)",
    "    for i in range(10):\n        print(i * 2)",
    "127 + 677 = 804",
]
for t in texts:
    print(our_encode(t) == enc.encode(t), "  ", repr(t))

**All `True`.** These 20 lines of Python *are* GPT-2's tokenizer. (`tiktoken` does exactly the same, just in fast Rust code.)

### 3b. Tokenizer quirks

Let's see how GPT-2 cuts different texts. Each colored box is one token:

In [ ]:
def show_gpt2_tokens(text):
    show_tokens([enc.decode([i]) for i in enc.encode(text)])

show_gpt2_tokens("Egg. I have an egg. EGG.")
show_gpt2_tokens("127 + 677 = 804 and 1275 + 6773 = 8048")
show_gpt2_tokens("How many r's are in strawberry?")
show_gpt2_tokens("def f(x):\n    if x > 0:\n        return x")

Things to notice:
- The same word gives different tokens depending on capitalization and on the space in front of it (`"Egg"`, `" egg"`, `" EGG"`).
- Numbers are cut into arbitrary pieces. No wonder LLMs used to be bad at arithmetic.
- `" strawberry"` is a **single** token. The model never sees its letters, which is why LLMs used to fail at counting the r's in it.
- In code, almost every space of the indentation is a **separate token**. That wastes a lot of tokens; newer tokenizers have tokens for long runs of spaces.

**Czech is expensive.** Compare the same sentence in English and Czech:

In [ ]:
english = "The capital of France is Paris."
czech = "Hlavní město Francie je Paříž."
show_gpt2_tokens(english)
show_gpt2_tokens(czech)

GPT-2 was trained mostly on English, so its merges are English. It does not even have a token for `ě` or `ř`: each is split into its 2 raw bytes, shown as `�`. Newer tokenizers have bigger vocabularies and handle other languages better:

In [ ]:
for name in ["gpt2", "cl100k_base", "o200k_base"]:      # used by GPT-2, GPT-4 and GPT-4o
    e = tiktoken.get_encoding(name)
    print(f"{name:12} vocabulary {e.n_vocab:>7,}    English: {len(e.encode(english)):2} tokens    Czech: {len(e.encode(czech)):2} tokens")

More tokens means a higher price (APIs charge per token), a context window that fills up faster, and a harder job for the model.

**Try it:** Tokenize a sentence of your own: in your mother tongue, with emoji, or a piece of code. You can also play with the [Tiktokenizer app](https://tiktokenizer.vercel.app).

In [ ]:
show_gpt2_tokens("Write your own sentence here.")

Finally, GPT-2 has one **special token**, `<|endoftext|>`, with id 50,256. During training it separates documents, so the model learns where a text ends. Chat models have more special tokens, e.g. to mark where the user's and the assistant's messages start.

In [ ]:
enc.encode("<|endoftext|>", allowed_special="all")

---
## 4. Meet GPT-2

The tokenizer is ready, now the model. We download GPT-2 small (124 million parameters, about 500 MB) from Hugging Face. *(On Colab you may see a warning about a missing `HF_TOKEN`. You can ignore it.)*

In [ ]:
from transformers import GPT2LMHeadModel

model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()                    # switch off dropout: we use the model, we don't train it
torch.set_grad_enabled(False)   # no training until section 10, so no gradients needed
print(model)

Read the printout from the top:
- `wte`: **w**ord **t**oken **e**mbedding, a table with one vector of 768 numbers for each of the 50,257 tokens (section 5)
- `wpe`: **w**ord **p**osition **e**mbedding, one vector for each of the 1,024 positions (section 5)
- `h`: 12 identical **blocks** (`GPT2Block`), each with `ln_1`, `attn`, `ln_2` and `mlp` (section 6)
- `ln_f`: a final LayerNorm
- `lm_head`: turns vectors back into 50,257 scores, one for each token (section 8)

Don't let `Conv1D` confuse you: despite the name, it is an ordinary linear layer.

How many parameters are where?

In [ ]:
def count(module):
    return sum(p.numel() for p in module.parameters())

block = model.transformer.h[0]
print(f"token embedding wte:    {count(model.transformer.wte):>12,}")
print(f"position embedding wpe: {count(model.transformer.wpe):>12,}")
print(f"one block:              {count(block):>12,}")
print(f"   attention:           {count(block.attn):>12,}")
print(f"   MLP:                 {count(block.mlp):>12,}")
print(f"all 12 blocks:          {count(model.transformer.h):>12,}")
print(f"final LayerNorm:        {count(model.transformer.ln_f):>12,}")
print(f"TOTAL:                  {count(model):>12,}")

Almost a third of the model is the embedding table, and in each block the MLP has twice as many parameters as the attention.

Where is `lm_head`? It does not add any parameters, because it **shares** its weights with `wte` (more in section 8):

In [ ]:
model.lm_head.weight is model.transformer.wte.weight

Let's ask GPT-2 what comes after our prompt. The model takes a **tensor** of token ids with shape `[batch, tokens]`:

In [ ]:
prompt = "The capital of France is"
ids = torch.tensor([enc.encode(prompt)])
print(ids, ids.shape)

logits = model(ids).logits
print(logits.shape)

The output has shape `[1, 5, 50257]`: for **each** of the 5 positions, one score for every token in the vocabulary, answering "what comes next after this position?". We only need the last position. Softmax turns the scores (**logits**) into probabilities:

In [ ]:
probs = F.softmax(logits[0, -1], dim=-1)
top = probs.topk(10)
for p, i in zip(top.values.tolist(), top.indices.tolist()):
    print(f"{p:6.1%}  {enc.decode([i])!r}")

Is `" Paris"` among them? In the next sections we open the box and compute these numbers ourselves.

---
## 5. Embeddings: from tokens to vectors

The first step inside the model is the simplest: a **lookup table**. `wte` has one row of 768 numbers for every token:

In [ ]:
wte = model.transformer.wte.weight.detach()     # detach() = just the numbers, without the training machinery
print(wte.shape)

paris = enc.encode(" Paris")[0]
print("token id of ' Paris':", paris)
print("its first 10 numbers:", wte[paris, :10])

These numbers were **learned** during training, just like the weights of the CNN in lesson 2. Tokens that are used in similar ways should get similar vectors. Let's check: for a given token, find the tokens whose vectors point in the most similar direction (**cosine similarity**):

In [ ]:
def neighbours(word, k=10):
    token_ids = enc.encode(word)
    assert len(token_ids) == 1, f"{word!r} is not a single token"
    unit = F.normalize(wte, dim=1)                      # every row scaled to length 1
    similarity = unit @ unit[token_ids[0]]              # cosine similarity with all 50,257 tokens
    best = similarity.topk(k + 1).indices[1:]           # skip the word itself
    return [enc.decode([i]) for i in best.tolist()]

for word in [" dog", " Paris", " Monday", " blue", " seven"]:
    print(f"{word!r:10} → {neighbours(word)}")

Look at the neighbours: other forms of the same word (plural, capitalized, without the space) and words of the same kind. This is the "directions carry meaning" idea from the video.

**Try it:** Try your own words. Remember the space: `" cat"` and `"cat"` are different tokens!

In [ ]:
neighbours(" cat")

**Position embeddings.** Attention (section 6) on its own does not know the order of tokens: "dog bites man" and "man bites dog" would look the same. So GPT-2 has a second table, `wpe`, with one learned vector for each position 0, 1, ..., 1023. Let's compare the position vectors with each other:

In [ ]:
wpe = model.transformer.wpe.weight.detach()
print(wpe.shape)

unit = F.normalize(wpe, dim=1)
plt.imshow(unit @ unit.T, cmap="viridis")      # cosine similarity of every pair of positions
plt.xlabel("position")
plt.ylabel("position")
plt.title("How similar are the position vectors?")
plt.colorbar()
plt.show()

The bright diagonal: each position is most similar to its **neighbouring** positions. Nobody told GPT-2 that position 10 is close to position 11. It learned that from data.

The input of the first block is simply the **sum**: token vector + position vector.

In [ ]:
positions = torch.arange(ids.shape[1])       # 0, 1, 2, 3, 4
x = wte[ids] + wpe[positions]                # shape [1, 5, 768]
print(x.shape)

Let's check this against GPT-2. With `output_hidden_states=True`, the model also returns the token vectors **between the blocks**: `hidden[0]` is the input of block 0, `hidden[1]` its output, and so on.

In [ ]:
hidden = model(ids, output_hidden_states=True).hidden_states
print(len(hidden), "tensors of shape", hidden[0].shape)
print("same as ours?", torch.allclose(x, hidden[0]))

---
## 6. A transformer block by hand

This is the heart of the transformer. All 12 blocks do the same thing:
```
x = x + attention(ln_1(x))     # tokens exchange information
x = x + mlp(ln_2(x))           # each token processes it on its own
```
The `x + ...` is the **residual connection**: a block does not replace the token vectors, it only **adds** something to them. (In the video this was the "residual stream".)

We will compute block 0 step by step with its real weights, and compare the result with GPT-2. First, what's inside:

In [ ]:
for name, p in block.named_parameters():
    print(f"{name:20} {tuple(p.shape)}")

`attn.c_attn.weight` has shape `(768, 2304)`, and 2304 = 3 × 768: the weights for **queries, keys and values** are packed into one matrix.

We will work with the input of block 0, without the batch dimension: 5 tokens × 768 numbers.

In [ ]:
x0 = hidden[0][0]           # input of block 0, shape (5, 768)
T = x0.shape[0]             # number of tokens
tokens = [enc.decode([i]) for i in ids[0].tolist()]
print(x0.shape, tokens)

### 6.1 LayerNorm

LayerNorm rescales every token vector to mean 0 and standard deviation 1, then multiplies it by learned weights and adds learned biases. It keeps the numbers in a sane range through all 12 blocks.

In [ ]:
def layer_norm(x, weight, bias, eps=1e-5):
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, keepdim=True, unbiased=False)
    return (x - mean) / torch.sqrt(var + eps) * weight + bias

h = layer_norm(x0, block.ln_1.weight, block.ln_1.bias)
print("same as GPT-2's ln_1?", torch.allclose(h, block.ln_1(x0), rtol=1e-4, atol=1e-4))

### 6.2 Attention: one head

Now the main act. Every token computes three vectors from its (normalized) vector `h`:
- **query**: "what am I looking for?"
- **key**: "what do I contain?"
- **value**: "what do I pass on to whoever looks at me?"

GPT-2 has 12 attention heads, and each of them works with vectors of 768 / 12 = 64 numbers. Let's compute **head 0**:

In [ ]:
qkv = h @ block.attn.c_attn.weight + block.attn.c_attn.bias    # (5, 2304)
q, k, v = qkv.split(768, dim=-1)                               # each (5, 768) = 12 heads × 64
print(q.shape, k.shape, v.shape)

q0, k0, v0 = q[:, :64], k[:, :64], v[:, :64]                   # head 0 = the first 64 numbers
print(q0.shape)

The **score** of token *i* for token *j* is the dot product of *i*'s query with *j*'s key. It is high when "what I am looking for" matches "what you contain". We divide by √64 = 8 to keep the numbers small:

In [ ]:
scores = q0 @ k0.T / math.sqrt(64)       # (5, 5): row = the token that looks, column = where it looks
print(scores)

**The causal mask.** GPT-2 predicts the next token, so a token must not look at tokens that come **after** it. That would be cheating! We set those scores to −∞, and softmax turns them into 0:

In [ ]:
mask = torch.tril(torch.ones(T, T, dtype=torch.bool))     # lower triangle = allowed
print(mask)

scores = scores.masked_fill(~mask, float("-inf"))
weights = F.softmax(scores, dim=-1)                        # each row now sums to 1
plot_attention(weights, tokens, "block 0, head 0")

Each row shows where one token looks. The first token can only look at itself. Finally, each token takes the **weighted average of the values** of the tokens it looks at:

In [ ]:
out0 = weights @ v0          # (5, 5) @ (5, 64) -> (5, 64)
print(out0.shape)

That's one head: every token now has 64 new numbers that summarize what it found in the earlier tokens.

### 6.3 All 12 heads at once

The other heads do the same, each with its own slice of 64 numbers. Instead of a loop, we reshape to `(12 heads, 5 tokens, 64)` and let PyTorch compute all 12 heads at once:

In [ ]:
def split_heads(t):                          # (5, 768) -> (12, 5, 64)
    return t.view(T, 12, 64).transpose(0, 1)

qh, kh, vh = split_heads(q), split_heads(k), split_heads(v)
scores = qh @ kh.transpose(-2, -1) / math.sqrt(64)       # (12, 5, 5)
weights = F.softmax(scores.masked_fill(~mask, float("-inf")), dim=-1)
out = weights @ vh                                       # (12, 5, 64)
print("head 0 same as before?", torch.allclose(out[0], out0, atol=1e-6))

Then we glue the heads back together (12 × 64 = 768) and mix them with one more linear layer, `c_proj`:

In [ ]:
out = out.transpose(0, 1).reshape(T, 768)                              # (5, 768)
attn_out = out @ block.attn.c_proj.weight + block.attn.c_proj.bias
print(attn_out.shape)

### 6.4 The MLP

The second half of the block works on each token **separately**. It expands the vector from 768 to 3,072 numbers, applies the **GELU** non-linearity (a smooth cousin of ReLU), and shrinks it back to 768. This is where much of the model's "knowledge" is believed to be stored.

In [ ]:
def mlp(h):
    expanded = F.gelu(h @ block.mlp.c_fc.weight + block.mlp.c_fc.bias, approximate="tanh")   # (5, 3072)
    return expanded @ block.mlp.c_proj.weight + block.mlp.c_proj.bias                         # (5, 768)

### 6.5 The whole block

Now we put it together, with both residual connections, and compare with the output of GPT-2's block 0:

In [ ]:
x1 = x0 + attn_out                                                  # residual connection 1
x1 = x1 + mlp(layer_norm(x1, block.ln_2.weight, block.ln_2.bias))   # residual connection 2

print("same as GPT-2's block 0?", torch.allclose(x1, hidden[1][0], rtol=1e-4, atol=1e-4))
print("largest difference:", (x1 - hidden[1][0]).abs().max().item())

🎉 The same numbers, up to tiny rounding errors. We have just computed a GPT-2 block by hand.

### 6.6 A reusable `Block` class

Let's pack everything into a proper PyTorch module, like `MyNetwork` in lesson 2. This time we use ordinary `nn.Linear` and `nn.LayerNorm` layers, so the class works for **any** size. In section 10 we will train it from scratch.

In [ ]:
class Block(nn.Module):
    def __init__(self, d, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.ln_1 = nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3 * d)      # queries, keys and values of all heads at once
        self.proj = nn.Linear(d, d)         # mixes the heads together
        self.ln_2 = nn.LayerNorm(d)
        self.fc = nn.Linear(d, 4 * d)       # MLP: expand ...
        self.fc_out = nn.Linear(4 * d, d)   # ... and shrink back

    def attention(self, h):
        B, T, d = h.shape                   # batch, tokens, vector size
        hs = d // self.n_heads              # head size (64 in GPT-2)
        q, k, v = self.qkv(h).split(d, dim=-1)
        # (B, T, d) -> (B, n_heads, T, hs): every head gets its own slice
        q, k, v = [t.view(B, T, self.n_heads, hs).transpose(1, 2) for t in (q, k, v)]
        scores = q @ k.transpose(-2, -1) / math.sqrt(hs)              # (B, n_heads, T, T)
        mask = torch.tril(torch.ones(T, T, dtype=torch.bool, device=h.device))
        weights = F.softmax(scores.masked_fill(~mask, float("-inf")), dim=-1)
        self.last_weights = weights.detach()                           # saved, so that we can look at it later
        out = (weights @ v).transpose(1, 2).reshape(B, T, d)           # glue the heads back together
        return self.proj(out)

    def mlp(self, h):
        return self.fc_out(F.gelu(self.fc(h), approximate="tanh"))

    def forward(self, x):
        x = x + self.attention(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

To load GPT-2's weights into our class, we copy them layer by layer. One detail: Hugging Face's `Conv1D` computes `x @ W`, while `nn.Linear` computes `x @ W.T`, so we copy the **transposed** matrix (`.T`).

In [ ]:
def copy_block_weights(ours, theirs):
    ours.ln_1.load_state_dict(theirs.ln_1.state_dict())
    ours.ln_2.load_state_dict(theirs.ln_2.state_dict())
    for our_layer, their_layer in [(ours.qkv, theirs.attn.c_attn), (ours.proj, theirs.attn.c_proj),
                                   (ours.fc, theirs.mlp.c_fc), (ours.fc_out, theirs.mlp.c_proj)]:
        our_layer.weight.copy_(their_layer.weight.T)
        our_layer.bias.copy_(their_layer.bias)

our_block = Block(d=768, n_heads=12)
copy_block_weights(our_block, block)
print("our Block == GPT-2's block 0?", torch.allclose(our_block(hidden[0]), hidden[1], rtol=1e-4, atol=1e-4))

### 6.7 The whole GPT-2 in 20 lines

A GPT is just: embeddings → a stack of blocks → final LayerNorm → **unembedding**. The last line of `forward` multiplies by the **same** table `wte` that turned tokens into vectors. This turns vectors back into one score per token (more in section 8).

In [ ]:
class GPT(nn.Module):
    def __init__(self, vocab_size, d, n_heads, n_layers, context):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, d)     # token embeddings
        self.wpe = nn.Embedding(context, d)        # position embeddings
        self.blocks = nn.ModuleList([Block(d, n_heads) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d)
        for p in self.parameters():                # starting weights: small random numbers (GPT-2's recipe)
            if p.dim() == 2:
                nn.init.normal_(p, std=0.02)

    def forward(self, ids):                        # ids: (B, T) token ids
        T = ids.shape[1]
        x = self.wte(ids) + self.wpe(torch.arange(T, device=ids.device))
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        return x @ self.wte.weight.T               # unembedding: (B, T, vocab_size) scores

Let's create it in GPT-2's size and copy all the weights:

In [ ]:
our_gpt2 = GPT(vocab_size=50257, d=768, n_heads=12, n_layers=12, context=1024)

hf = model.transformer
our_gpt2.wte.weight.copy_(hf.wte.weight)
our_gpt2.wpe.weight.copy_(hf.wpe.weight)
our_gpt2.ln_f.load_state_dict(hf.ln_f.state_dict())
for ours, theirs in zip(our_gpt2.blocks, hf.h):
    copy_block_weights(ours, theirs)

print(f"parameters: {count(our_gpt2):,}")
our_logits = our_gpt2(ids)
print("our GPT-2 == real GPT-2?", torch.allclose(our_logits, logits, rtol=1e-4, atol=1e-4))
print("largest difference:", (our_logits - logits).abs().max().item())

🎉 **Our ~60 lines of PyTorch compute the same thing as GPT-2.** From now on we use `our_gpt2`.

---
## 7. What do attention heads look at?

GPT-2 has 12 blocks × 12 heads = 144 heads. Our `Block` saves its attention weights in `last_weights`, so let's run a sentence and look:

In [ ]:
def get_attention(ids):
    """Run our GPT-2 and collect the attention weights of all heads: (12 blocks, 12 heads, T, T)."""
    our_gpt2(ids)
    return torch.stack([b.last_weights[0] for b in our_gpt2.blocks])

sentence = "When Mary and John went to the store, John gave a drink to Mary."
sent_ids = torch.tensor([enc.encode(sentence)])
sent_tokens = [enc.decode([i]) for i in sent_ids[0].tolist()]
attention = get_attention(sent_ids)
print(attention.shape)

plot_heads(attention[0], "block 0")
plot_heads(attention[5], "block 5")

Many heads (especially in later blocks) put most of their weight on the **very first token**: a bright first column. This is called an **attention sink**: when a head finds nothing useful to look at, it "parks" its attention there.

Instead of staring at 144 pictures, let's measure two simple patterns for every head:
- how much each token looks at the token **right before it**
- how much each token looks at the **first** token

In [ ]:
prev_score = attention.diagonal(offset=-1, dim1=-2, dim2=-1).mean(-1)   # (12, 12): weight on the previous token
first_score = attention[:, :, 1:, 0].mean(-1)                           # (12, 12): weight on the first token

plot_head_scores(prev_score, "attention to the previous token")
plot_head_scores(first_score, "attention to the first token")

In [ ]:
layer, head = divmod(prev_score.argmax().item(), 12)
print(f"the strongest previous-token head: block {layer}, head {head}")
plot_attention(attention[layer, head], sent_tokens, f"block {layer}, head {head}")

**Induction heads.** A famous discovery: some heads help the model **continue patterns**. If the text contains "... Harry Potter ... Harry", an induction head looks at the token that followed the previous "Harry" (i.e. "Potter"), which helps predict it again.

To find them, we feed GPT-2 20 random tokens, repeated twice. In the second copy, an induction head looks at the token **one after** the same token in the first copy, i.e. 19 positions back:

In [ ]:
torch.manual_seed(0)
n = 20
random_ids = torch.randint(1000, 10000, (1, n))
rep_ids = torch.cat([random_ids, random_ids], dim=1)     # 40 tokens: the same 20 twice
rep_attention = get_attention(rep_ids)

# a token in the second copy (from position 20 on) looking 19 positions back
induction_score = rep_attention.diagonal(offset=-(n - 1), dim1=-2, dim2=-1)[:, :, 1:].mean(-1)
plot_head_scores(induction_score, "induction score")

layer, head = divmod(induction_score.argmax().item(), 12)
print(f"the strongest induction head: block {layer}, head {head}")
rep_tokens = [enc.decode([i]) for i in rep_ids[0].tolist()]
plot_attention(rep_attention[layer, head], rep_tokens, f"block {layer}, head {head}")

In the second half you can see a diagonal stripe: each token looks at the place right after its earlier copy.

**Try it:** Look at other heads with `plot_attention(attention[block, head], sent_tokens)`. Can you find a head that connects the second `" John"` or `" Mary"` with the first one?

In [ ]:
plot_attention(attention[9, 9], sent_tokens, "block 9, head 9")

---
## 8. From vectors back to text

After the last block and `ln_f`, every token vector is multiplied by the transposed embedding table `wte`. This gives 50,257 scores (**logits**), one for each token, and softmax turns them into probabilities. Using the same table at the start and at the end is called **weight tying**: a vector that points in the same direction as the embedding of `" Paris"` gets a high score for `" Paris"`.

In [ ]:
def plot_next_token(prompt, k=10):
    ids = torch.tensor([enc.encode(prompt)])
    probs = F.softmax(our_gpt2(ids)[0, -1], dim=-1)      # scores after the LAST token
    top = probs.topk(k)
    labels = [repr(enc.decode([i])) for i in top.indices.tolist()]
    plt.figure(figsize=(6, 0.35 * k + 1))
    plt.barh(labels[::-1], top.values.tolist()[::-1])
    plt.xlabel("probability")
    plt.title(f"{prompt} ...")
    plt.show()

plot_next_token("The capital of France is")

**Try it:** Try your own prompts. Remember that GPT-2 is **not a chatbot**: it just continues text. If you ask a question, it will happily continue with more questions.

In [ ]:
plot_next_token("My favourite programming language is")

**Generating text** is just a loop: predict the next token, pick one, append it, repeat.

*How* we pick matters. The **temperature** divides the logits before softmax:
- temperature → 0: always the most likely token (boring, often repeats itself)
- temperature = 1: sample from the model's own probabilities
- temperature > 1: flatter probabilities, more surprising choices (and more nonsense)

In [ ]:
logits_last = our_gpt2(torch.tensor([enc.encode(prompt)]))[0, -1]
top_ids = logits_last.topk(8).indices
labels = [repr(enc.decode([i])) for i in top_ids.tolist()]

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), sharey=True)
for ax, temperature in zip(axes, [0.5, 1.0, 2.0]):
    probs = F.softmax(logits_last / temperature, dim=-1)
    ax.barh(labels[::-1], probs[top_ids].tolist()[::-1])
    ax.set_title(f"temperature {temperature}")
    ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
@torch.no_grad()
def generate(model, ids, n_new, temperature=1.0, context=1024):
    """Append n_new tokens to ids (a tensor of shape [1, T]), one at a time."""
    for _ in range(n_new):
        logits = model(ids[:, -context:])[:, -1, :]       # scores for the next token (the model sees at most `context` tokens)
        if temperature == 0:
            next_id = logits.argmax(dim=-1, keepdim=True)              # greedy: the most likely token
        else:
            probs = F.softmax(logits / temperature, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)          # a random draw according to probs
        ids = torch.cat([ids, next_id], dim=1)
    return ids

torch.manual_seed(42)
start = torch.tensor([enc.encode("The capital of France is")])
for temperature in [0, 0.7, 1.5]:
    out = generate(our_gpt2, start, 30, temperature=temperature)
    print(f"--- temperature {temperature} ---")
    print(enc.decode(out[0].tolist()))

**Try it:** Your own prompt, other temperatures, longer texts.

In [ ]:
# your code here

---
## 9. Bonus: the logit lens

What does GPT-2 "think" halfway through? We can apply the final LayerNorm and the unembedding after **every** block, not just after the last one, and see which token would win at that point. This trick is called the **logit lens**.

In [ ]:
ids = torch.tensor([enc.encode("The capital of France is")])
x = our_gpt2.wte(ids) + our_gpt2.wpe(torch.arange(ids.shape[1]))
for i, b in enumerate(our_gpt2.blocks):
    x = b(x)
    probs = F.softmax(our_gpt2.ln_f(x[0, -1]) @ our_gpt2.wte.weight.T, dim=-1)
    best = enc.decode([probs.argmax().item()])
    print(f"after block {i:2}:  top token {best!r:14}  P(' Paris') = {probs[paris].item():.4f}")

In which block does `" Paris"` start to win? Try other prompts, e.g. `"The Eiffel Tower is in the city of"`.

---
## 10. Training a mini GPT from scratch

So far, GPT-2's weights came from OpenAI's training. Now we train our **own** GPT from random weights, using exactly the same `Block` and `GPT` classes. It is much smaller, so it trains in a few minutes:

| | GPT-2 small | our mini GPT |
|---|---|---|
| blocks | 12 | 4 |
| vector size | 768 | 128 |
| heads | 12 | 4 |
| context | 1,024 tokens | 128 characters |
| vocabulary | 50,257 BPE tokens | 65 characters |
| parameters | 124 million | 0.8 million |
| training data | 40 GB of web pages | 1 MB of Shakespeare |

To keep it simple, our tokens are single **characters**. (With GPT-2's 50,257 tokens, the embedding table alone would have 6.4 million parameters, 8× more than the rest of our tiny model.)

**This section needs a GPU.** If `device` below prints `cpu`, the training runs only 300 steps as a short demo. To get a GPU now: *Runtime → Change runtime type → T4 GPU*, then click into this cell and choose *Runtime → Run before*.

In [ ]:
torch.set_grad_enabled(True)        # we switched gradients off in section 4; training needs them
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

with open("input.txt", encoding="utf-8") as f:     # tiny Shakespeare, downloaded in section 2
    text = f.read()

chars = sorted(set(text))
vocab_size = len(chars)
print(vocab_size, "different characters:", repr("".join(chars)))

In [ ]:
stoi = {ch: i for i, ch in enumerate(chars)}     # character -> id
itos = {i: ch for i, ch in enumerate(chars)}     # id -> character

def encode_chars(s):
    return [stoi[ch] for ch in s]

def decode_chars(ids):
    return "".join(itos[i] for i in ids)

print(encode_chars("To be, or not to be"))
print(decode_chars(encode_chars("To be, or not to be")))

data = torch.tensor(encode_chars(text))
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]        # first 90% for training, the last 10% for validation
print(f"{len(train_data):,} training and {len(val_data):,} validation characters")

**Training examples.** We cut random windows of 129 characters out of the text. The input is the first 128 characters and the target is the same window **shifted by one**. So one window contains 128 examples: after `"F"` comes `"i"`, after `"Fi"` comes `"r"`, after `"Fir"` comes `"s"`, ... Thanks to the causal mask, the model learns all of them at once.

In [ ]:
context = 128
batch_size = 64

def get_batch(split):
    d = train_data if split == "train" else val_data
    starts = torch.randint(len(d) - context - 1, (batch_size,))
    x = torch.stack([d[s:s + context] for s in starts])
    y = torch.stack([d[s + 1:s + context + 1] for s in starts])     # the same, shifted by one
    return x.to(device), y.to(device)

x, y = get_batch("train")
print(x.shape, y.shape)
print("input: ", repr(decode_chars(x[0, :40].tolist())))
print("target:", repr(decode_chars(y[0, :40].tolist())))

Now the model: the same `GPT` class as before, just smaller and with random weights. The loss is the **cross-entropy** of the true next character, averaged over all positions, exactly like the classification loss in lesson 2.

In [ ]:
torch.manual_seed(1337)
mini = GPT(vocab_size=vocab_size, d=128, n_heads=4, n_layers=4, context=context).to(device)
print(f"parameters: {count(mini):,}")

def compute_loss(model, x, y):
    logits = model(x)                                                 # (B, T, vocab_size)
    return F.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1))

x, y = get_batch("train")
print(f"loss before training: {compute_loss(mini, x, y).item():.3f}    ln(vocab_size) = {math.log(vocab_size):.3f}")

start = torch.zeros((1, 1), dtype=torch.long, device=device)          # character 0 is the newline "\n"
print(decode_chars(generate(mini, start, 200, context=context)[0].tolist()))

Before training, the loss is about ln(65) ≈ 4.17: the model gives every character the same chance, and its "Shakespeare" is random characters.

The training loop has the **same 5 steps** as in lesson 2. Every 250 steps we measure the loss on the training and validation data, and every 1,000 steps we print a short sample:

In [ ]:
@torch.no_grad()
def estimate_loss(model, n_batches=20):
    """Average loss on a few random batches, for training and validation data."""
    result = {}
    for split in ["train", "val"]:
        losses = [compute_loss(model, *get_batch(split)).item() for _ in range(n_batches)]
        result[split] = sum(losses) / len(losses)
    return result

In [ ]:
import time

optimizer = torch.optim.AdamW(mini.parameters(), lr=1e-3)
steps = 3000 if device == "cuda" else 300        # on a CPU only a short demo
history = []
t0 = time.time()

for step in range(1, steps + 1):
    x, y = get_batch("train")
    loss = compute_loss(mini, x, y)      # 1. FORWARD + 2. LOSS
    optimizer.zero_grad()                # 5. RESET
    loss.backward()                      # 3. BACKWARD
    optimizer.step()                     # 4. UPDATE

    if step % 250 == 0 or step == steps:
        losses = estimate_loss(mini)
        history.append((step, losses["train"], losses["val"]))
        print(f"step {step:5}:  train loss {losses['train']:.3f}   validation loss {losses['val']:.3f}   ({time.time() - t0:.0f} s)")
    if step % 1000 == 0:
        print("   " + decode_chars(generate(mini, start, 120, context=context)[0].tolist()).replace("\n", "\n   "))

In [ ]:
done, train_losses, val_losses = zip(*history)
plt.plot(done, train_losses, label="training")
plt.plot(done, val_losses, label="validation")
plt.xlabel("step")
plt.ylabel("loss")
plt.legend()
plt.show()

And now a longer text from our own little GPT:

In [ ]:
torch.manual_seed(42)
print(decode_chars(generate(mini, start, 1000, temperature=0.8, context=context)[0].tolist()))

It is not Shakespeare, but look at what 0.8 million parameters learned in a few minutes, from nothing but predicting the next character: names of characters in capitals followed by a colon, line breaks, punctuation, and many real English words.

GPT-2 has the same architecture, just 150× more parameters and 40,000× more training data.

**Try it:**
- Train for more steps, or make the model bigger (e.g. `d=192, n_layers=6`). Does the validation loss go down? Watch the gap between the training and validation loss.
- Train on your own text: upload a `.txt` file (folder icon on the left) and change the file name in the first cell of this section.
- Generate with `temperature=0.5` and `temperature=1.5`.

---
## 11. Summary

| step | in GPT-2 | what we did |
|---|---|---|
| text → tokens | BPE: 256 bytes + 50,000 merges, regex pre-split | rebuilt the tokenizer in ~20 lines, same tokens as `tiktoken` |
| tokens → vectors | `wte` (50,257 × 768) + `wpe` (1,024 × 768) | nearest neighbours, similarity of positions |
| 12 × block | `x + attention(ln_1(x))`, then `x + mlp(ln_2(x))` | rebuilt `Block` and `GPT`, same numbers as GPT-2 |
| attention | 12 heads, queries · keys, causal mask, weighted average of values | previous-token, first-token and induction heads |
| vectors → next token | `ln_f`, then multiply by `wte` again (weight tying), softmax | top-10 tokens, generation, temperature, logit lens |
| training | cross-entropy of the next token | a 0.8M-parameter GPT trained on Shakespeare, same classes |

**New words:** token, Byte Pair Encoding (BPE), merge, vocabulary, embedding, position embedding, residual connection, LayerNorm, query, key, value, attention head, causal mask, MLP, GELU, logits, unembedding, weight tying, temperature.

### Exercises
1. Tokenize the same paragraph in Czech, English and one more language with `gpt2` and `o200k_base`. Which language is the most "expensive"?
2. In section 7, find a head that does something else you can describe, e.g. one that connects a name with its earlier occurrence.
3. In the `Block` class, remove the causal mask, so that every token can see every token (this is how BERT works), and run section 8 again. What happens to GPT-2's predictions, and why?
4. Train the mini GPT with `n_layers=1` and with `n_layers=8`. How does the validation loss change? And the training time?

### Further reading
- Interactive GPT-2 in your browser: [Transformer Explainer](https://poloclub.github.io/transformer-explainer/), [LLM Visualization](https://bbycroft.net/llm)
- Jay Alammar: [The Illustrated GPT-2](https://jalammar.github.io/illustrated-gpt2/)
- Andrej Karpathy: [Let's build the GPT Tokenizer](https://www.youtube.com/watch?v=zduSFxRajkE), [Let's build GPT](https://www.youtube.com/watch?v=kCc8FmEb1nY), [Let's reproduce GPT-2 (124M)](https://www.youtube.com/watch?v=l8pRSuU81PU)
- Sebastian Raschka: [From GPT-2 to gpt-oss](https://sebastianraschka.com/blog/2025/from-gpt-2-to-gpt-oss.html): what has changed in LLM architectures since GPT-2